In [ ]:
import os
import cv2
import numpy as np
import joblib
import matplotlib.pyplot as plt
from tqdm import tqdm

test_dir = r"E:\Road_Quality\Frame_Num\MatchedFrames"
model_dir = r"E:\Road_Quality\model\CSI_percentile_linear"

model = joblib.load(os.path.join(model_dir, "model.joblib"))

def list_jpgs(folder):
    return [os.path.join(folder, x) for x in os.listdir(folder) if x.lower().endswith(".jpg")]

def compute_csi(gray):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(np.std(lap))

def compute_lcr(gray):
    edges = cv2.Canny(gray, 100, 200)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50, minLineLength=30, maxLineGap=10)
    if lines is None:
        return 0.0
    tot = 0.0
    for l in lines:
        x1, y1, x2, y2 = l[0]
        tot += float(np.sqrt((x2-x1)**2 + (y2-y1)**2))
    return float(tot / (edges.size + 1e-6))

paths = list_jpgs(test_dir)

X = []
for p in tqdm(paths, desc="Testing CSI-linear on MatchedFrames"):
    g = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if g is None:
        continue
    csi = compute_csi(g)
    lcr = compute_lcr(g)
    mu = float(np.mean(g))
    sd = float(np.std(g))
    X.append([csi, lcr, mu, sd])

X = np.array(X, dtype=np.float32)

pred = model.predict(X)
pred = np.clip(pred, 0.0, 1.0)

# Plot distribution (bar chart bins)
bins = np.linspace(0, 1.0, 11)
hist, edges = np.histogram(pred, bins=bins)
centers = 0.5 * (edges[:-1] + edges[1:])

plt.figure()
plt.bar(centers, hist, width=(edges[1]-edges[0]) * 0.9)
plt.xlabel("Predicted CSI Percentile (0..1)")
plt.ylabel("Count")
plt.title("CSI Percentile Predictions (Linear Model) on MatchedFrames")
plt.tight_layout()
plt.show()

print("CSI predicted percentile quartiles:", np.quantile(pred, [0, 0.25, 0.5, 0.75, 1.0]))


Testing CSI-linear on MatchedFrames:  74%|██████████████████████████████▍          | 2483/3339 [04:03<01:38,  8.70it/s]

In [ ]:
import os
import cv2
import numpy as np
import joblib
import matplotlib.pyplot as plt
from tqdm import tqdm

test_dir = r"E:\Road_Quality\Frame_Num\MatchedFrames"
model_dir = r"E:\Road_Quality\model\LCR_percentile_linear"

model = joblib.load(os.path.join(model_dir, "model.joblib"))

def list_jpgs(folder):
    return [os.path.join(folder, x) for x in os.listdir(folder) if x.lower().endswith(".jpg")]

def compute_csi(gray):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(np.std(lap))

def compute_lcr(gray):
    edges = cv2.Canny(gray, 100, 200)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50, minLineLength=30, maxLineGap=10)
    if lines is None:
        return 0.0
    tot = 0.0
    for l in lines:
        x1, y1, x2, y2 = l[0]
        tot += float(np.sqrt((x2-x1)**2 + (y2-y1)**2))
    return float(tot / (edges.size + 1e-6))

paths = list_jpgs(test_dir)

X = []
for p in tqdm(paths, desc="Testing LCR-linear on MatchedFrames"):
    g = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if g is None:
        continue
    csi = compute_csi(g)
    lcr = compute_lcr(g)
    mu = float(np.mean(g))
    sd = float(np.std(g))
    X.append([csi, lcr, mu, sd])

X = np.array(X, dtype=np.float32)

pred = model.predict(X)
pred = np.clip(pred, 0.0, 1.0)

bins = np.linspace(0, 1.0, 11)
hist, edges = np.histogram(pred, bins=bins)
centers = 0.5 * (edges[:-1] + edges[1:])

plt.figure()
plt.bar(centers, hist, width=(edges[1]-edges[0]) * 0.9)
plt.xlabel("Predicted LCR Percentile (0..1)")
plt.ylabel("Count")
plt.title("LCR Percentile Predictions (Linear Model) on MatchedFrames")
plt.tight_layout()
plt.show()

print("LCR predicted percentile quartiles:", np.quantile(pred, [0, 0.25, 0.5, 0.75, 1.0]))
